# FoS v0.7.4 Evaluation Benchmark — Colab workflow

This notebook profiles real target-pair caches, enriches ChEMBL document metadata, builds an action-space-aware retrospective release, audits leakage, and freezes the release.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
PROJECT_ZIP = Path('/content/drive/MyDrive/FoS-feature-stage-c-v074-evaluation-benchmark.zip')
CACHE_BUNDLE = Path('/content/drive/MyDrive/FoS_cache_bundle.zip')  # adjust
WORKDIR = Path('/content/FoS_v074')

In [ ]:
!rm -rf "$WORKDIR"
!mkdir -p "$WORKDIR"
!unzip -q "$PROJECT_ZIP" -d "$WORKDIR"

import os, glob
PROJECT = Path(glob.glob(str(WORKDIR / 'FoS-feature-stage-c-v074-evaluation-benchmark'))[0])
os.chdir(PROJECT)
print(PROJECT)

In [ ]:
!pip -q install -e .
!python -V
!pytest -q

## Restore Stage A evidence and raw ChEMBL caches

The cache bundle should contain `data/cache/evidence_live` and ideally `data/raw/chembl/cache`. Adjust the extraction path to your bundle layout.

In [ ]:
!unzip -q -o "$CACHE_BUNDLE" -d "$PROJECT"
!find data/cache/evidence_live/pairs -maxdepth 1 -mindepth 1 -type d | head

## Candidate pair density audit

Add or remove target pairs after checking the available pair-cache directories.

In [ ]:
!PYTHONPATH=src python scripts/profile_target_pairs.py \
  --cache-dir data/cache/evidence_live \
  --pair CHEMBL203:CHEMBL1824 \
  --raw-chembl-cache-dir data/raw/chembl/cache \
  --output-dir outputs/evaluation_pair_profiles \
  --write-enriched-paired

import pandas as pd
profile = pd.read_csv('outputs/evaluation_pair_profiles/pair_profile.csv')
display(profile)

## Fetch document publication years

This cell uses the ChEMBL API and may take time on a cold cache. Run it once and back up the CSV.

In [ ]:
!PYTHONPATH=src python scripts/fetch_chembl_document_metadata.py \
  --raw-chembl-cache-dir data/raw/chembl/cache \
  --output evaluation/document_metadata.csv

display(pd.read_csv('evaluation/document_metadata.csv').head())

## Re-run pair audit with years and select a split cutoff

In [ ]:
!PYTHONPATH=src python scripts/profile_target_pairs.py \
  --cache-dir data/cache/evidence_live \
  --pair CHEMBL203:CHEMBL1824 \
  --raw-chembl-cache-dir data/raw/chembl/cache \
  --document-metadata evaluation/document_metadata.csv \
  --output-dir outputs/evaluation_pair_profiles_with_years \
  --write-enriched-paired

display(pd.read_csv('outputs/evaluation_pair_profiles_with_years/pair_profile.csv'))

## Build an action-space-aware development release

Set `CUTOFF_YEAR` only after reviewing the document/year distribution. EGFR/HER2 is shown as a pipeline example; benchmark pair selection should follow the density audit.

In [ ]:
ON_TARGET = 'CHEMBL203'
OFF_TARGET = 'CHEMBL1824'
CUTOFF_YEAR = 2020
RELEASE_DIR = Path(f'evaluation/releases/{ON_TARGET}_{OFF_TARGET}_dev')

!rm -rf "$RELEASE_DIR"
!PYTHONPATH=src python scripts/build_measured_benchmark_v2.py \
  --cache-dir data/cache/evidence_live \
  --on-target "$ON_TARGET" \
  --off-target "$OFF_TARGET" \
  --split development \
  --split-strategy document_time \
  --cutoff-year "$CUTOFF_YEAR" \
  --raw-chembl-cache-dir data/raw/chembl/cache \
  --document-metadata evaluation/document_metadata.csv \
  --max-depth 2 \
  --max-positive 20 \
  --max-negative 10 \
  --output-dir "$RELEASE_DIR"

## Inspect the release and freeze it

In [ ]:
import json
manifest = json.loads((RELEASE_DIR/'manifests/split_manifest.json').read_text())
audit = json.loads((RELEASE_DIR/'manifests/leakage_audit.json').read_text())
print(json.dumps(manifest, indent=2))
print('audit passed:', audit['passed'])
display(pd.read_csv(RELEASE_DIR/'manifests/episode_selection_audit.csv'))

!PYTHONPATH=src python scripts/freeze_evaluation_release.py "$RELEASE_DIR"

## Back up the frozen release

In [ ]:
BACKUP = Path('/content/drive/MyDrive/FoS_evaluation_releases')
BACKUP.mkdir(parents=True, exist_ok=True)
archive = BACKUP / f'{RELEASE_DIR.name}.zip'
!cd "$RELEASE_DIR.parent" && zip -qr "$archive" "$RELEASE_DIR.name"
print(archive)